In [1]:
import numpy as np                        # Μαθηματική/αριθμητική βιβλιοθήκη
import scipy.io.wavfile as wav            # Βιβλιοθήκη ανάγνωσης/εγγραφής ήχου
from IPython.display import Audio         # Βιβλιοθήκη παιξίματος αρχείων ήχου
%matplotlib inline

## Άσκηση - Προσθήκη Ηχούς σε ένα σήμα

Δεν υπάρχει αμφιβολία ότι οι διαφορικές εξισώσεις κυριαρχούν στη μοντελοποίηση συστημάτων συνεχούς χρόνου. Όμως κάποια χρήσιμα συστήματα **δεν** περιγράφονται απαραίτητα από τέτοιες. Ας δούμε ένα τέτοιο σύστημα σε αυτήν την άσκηση. 

Κατά την παραγωγή και καταγραφή ήχου σε ένα χώρο όπου υπάρχουν πολλές ανακλάσεις, εμπόδια, κλπ., το σήμα του ήχου καταγράφεται ως άθροισμα πολλών διαφορετικών "εκδόσεων" (καθυστερήσεων) του σήματος που επιστρέφουν μαζί στο μικρόφωνο, ως ηχώ, κατά την καταγραφή. Μπορούμε να μοντελοποιήσουμε την ηχώ ως ένα ΓΧΑ σύστημα, το οποίο περιγράφεται από τη σχέση:

$$ y(t) = x(t) + ax(t-t_d) $$

με $a$ το πλάτος της ηχούς και $t_d$ τη καθυστέρησή της στο χρόνο, δηλ. τη χρονική στιγμή που εμφανίζεται στο ηχογραφημένο σήμα.

Δοκιμάστε να βρείτε στο χαρτί σας την κρουστική απόκριση, $h(t)$, του ΓΧΑ συστήματος, δεδομένου ότι γνωρίζετε ότι η κρουστική απόκριση δίνεται ως η έξοδος ενός συστήματος για είσοδο $$x(t) = δ(t)$$ 

Θα πρέπει να πάρετε την απάντηση $$h(t) = \delta(t) + a \delta(t-t_d)$$

Θα μπορούσαμε να προσθέσουμε κι άλλα αντίγραφα του σήματος, σε διαφορετικές χρονικές στιγμές και με διαφορετικούς συντελεστές (εντάσεις). Όπως μπορείτε εύκολα να καταλάβετε, ένα τέτοιο σύστημα θα είναι της μορφής:
$$ y(t) = x(t) + \sum_{i=1}^N a_i x(t-t_i) $$

Αν δοκιμάσετε να βρείτε στο χαρτί σας την κρουστική απόκριση, $h(t)$, του παραπάνω ΓΧΑ συστήματος, θα καταλήξετε στην απάντηση

$$ h(t) = \delta(t) + \sum_{i=1}^N a_i \delta(t-t_i) $$

Με λίγα ακόμα μαθηματικά που δεν είναι του παρόντος, μπορούμε να δείξουμε ότι η παραπάνω κρουστική απόκριση μπορεί να δειγματοληπτηθεί κι αυτή (δείτε την άσκηση $\texttt{Ex-DifferentialEquations}$), και να μας δώσει την κρουστική απόκριση *διακριτού χρόνου* 

$$ h(n) = \delta(n) + \sum_{i=1}^N a_i \delta(n-n_i) $$

με τη συνάρτηση $\delta(n)$ να ονομάζεται "Δέλτα του Kronecker", και να είναι ένα πολύ ωραίο σήμα - σε αντίθεση με τη γενικευμένη συνάρτηση $\delta(t)$, της οποίας γνωρίζετε τις "ιδιοτροπίες" - της μορφής

$$\delta(n) =
\begin{cases}
1, & n = 0 \\
0, & n \neq 0
\end{cases} $$

Θα υλοποιήσουμε την κρουστική απόκριση του συστήματος παραγωγής ηχούς και θα εφαρμόσουμε το σύστημα επάνω σε ένα οποιοδήποτε ηχητικό σήμα εισόδου με χρήση της πράξης της συνέλιξης! 

Συμπληρώστε την επόμενη συνάρτηση στην ``Python`` υλοποιώντας τη σχέση $h(n)$ παραπάνω:

In [2]:
def echo_filter(signal, times, attenuations, fs):
    """
    Συνάρτηση που φιλτράρει το σήμα εισόδου signal και παράγει την έξοδο y_echo

    :param signal: σήμα εισόδου που θέλετε να βάλετε ηχώ
    :param times: διάνυσμα που περιέχει τις χρονικές στιγμές - σε δευτερόλεπτα - που θέλουμε να ξεκινά μια ηχώ επάνω στο σήμα εισόδου
    :param attenuations: διάνυσμα που περιέχει το πλάτος της κάθε ηχούς στις αντίστοιχες χρονικές στιγμές που έχετε ορίσει στο διάνυσμα times επάνω
    :param fs: συχνότητα δειγματοληψίας του σήματος, επιστρέφεται από την audioread όταν διαβάζετε ένα .WAV αρχείο
    :return: y_echo = έξοδος του ΓΧΑ συστήματος (σήμα με ηχώ)
    """
    h = np.zeros(shape=len(signal))
    samples = np.zeros(shape=len(times))
    
    h[0] = 1

    for i in range(len(times)):
        samples[i] = times[i] * fs
        h[int(samples[i])] = attenuations[i]

    y_echo = np.convolve(signal, h) # (μπορείτε να χρησιμοποιήσετε την "convolve" από τη NumPy για να κάνετε συνέλιξη την είσοδο με την κρουστική απόκριση - δείτε το documentation της)

    return y_echo

---
Μια μικρή επεξήγηση για τη 6η γραμμή του κώδικα (``samples[i]=...``). Όπως ήδη ξέρετε - αν έχετε δουλέψει τις ασκήσεις του προηγούμενου Κεφαλαίου - όλα τα σήματα που επεξεργαζόμαστε στον υπολογιστή είναι διακριτού χρόνου, δηλ. ορισμένα για συγκεκριμένες χρονικές τιμές (και όχι για κάθε $t$), εσείς πρέπει αρχικά να ορίσετε τις τιμές του διανύσματος times που θέλετε να ακούγεται η ηχώ (σε δευτερόλεπτα), και να μετατρέψετε στη γραμμή 6 κάθε τιμή του διανύσματος αυτού σε ακέραιες τιμές, δηλ. σε δείγματα. 

Αυτό γίνεται εύκολα αν λάβετε υπόψη σας ότι η συχνότητα δειγματοληψίας $f_s$ ενός σήματος σας λέει ότι σε ένα δευτερόλεπτο ηχογράφησης έχουν παρθεί και αποθηκευτεί $f_s$ δείγματα (τιμές) του σήματος στον υπολογιστή. ΄Αρα, π.χ., η χρονική στιγμή $t_0 = 0.5$ s αντιστοιχεί στο δείγμα διακριτού χρόνου $f_s/2$. Σε ποια δείγματα αντιστοιχούν οι δικές σας χρονικές στιγμές της ηχούς που (θα) ορίσετε στο διάνυσμα ``times``; Αυτή η μετατροπή γράφεται στη γραμμή 6.

---

Μπορείτε να χρησιμοποιήσετε ένα οποιοδήποτε σήμα φωνής/μουσικής σε μορφή .WAV για να ελέγξετε τη λειτουργία του συστήματός σας. Απλά φροντίστε να μην είναι πολύ μεγάλης διάρκειας για να μην αργήσει πολύ ή κρασάρει το πρόγραμμα. 

Προς βοήθειά σας, μια εντολή για να φορτώσετε ένα .WAV σήμα με όνομα ``furelise_cut.wav`` μέσω της SciPy είναι:

In [3]:
fs, sig = wav.read('./files/furelise_cut.wav')
Audio(sig, rate=fs)

C:\Users\kafge\AppData\Local\Temp\ipykernel_832\946429060.py:1: WavFileWarning: Chunk (non-data) not understood, skipping it.
  fs, sig = wav.read('./files/furelise_cut.wav')


Έπειτα από το φιλτράρισμα του παραπάνω σήματός (ή όποιου δικού σας θέλετε να βάλετε) με κάποιες επιλεγμένες τιμές για το ``times`` και ``attenuation``, μπορούμε να το ακούσουμε.

In [4]:
times1 = np.asarray([2.1]) # μια ηχώ στο δευτερόλεπτο 2.1
attenuations1 = np.asarray([0.4]) # η ηχώ στο 2.1 s θα έχει συντελεστή 0.8 (λίγο πιο "χαμηλή" ένταση από την ίδια την ηχογράφηση),

echoed_signal = echo_filter(sig, times1, attenuations1, fs) # INSERT CODE HERE (καλέστε τη συνάρτηση που γράψατε παραπάνω)

Audio(echoed_signal, rate=fs)

Άλλη μια, λίγο πιο σύνθετη περίπτωση.

In [5]:
times2 = np.asarray([2.1, 5.6]) # μια ηχώ στο δευτερόλεπτο 2.1 και στο δευτερόλεπτο 5.6
attenuations2 = np.asarray([0.5, 0.25]) # η ηχώ στο 2.1 s θα έχει συντελεστή 0.5 (το μισό της έντασης της αρχικής ηχογράφησης),
                                      # ενώ αυτή στο 5.6 s θα έχει συντελεστή 0.25 (το 1/4 της έντασης της αρχικής ηχογράφησης)

echoed_signal = echo_filter(sig, times2, attenuations2, fs) # INSERT CODE HERE (καλέστε τη συνάρτηση που γράψατε παραπάνω)

Audio(echoed_signal, rate=fs)

Πρέπει να ακούσετε ότι όντως προστέθηκε ηχώ (echo) στο σήμα σας - εάν τα έχετε κάνει όλα σωστά! 😊

---
---